In [5]:
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
import time

In [6]:
X_train = pd.read_csv("../../../data/processed/exp_1/X_train_tree.csv")
X_test  = pd.read_csv("../../../data/processed/exp_1/X_test_tree.csv")
y_train = pd.read_csv("../../../data/processed/exp_1/y_train.csv").squeeze()
y_test  = pd.read_csv("../../../data/processed/exp_1/y_test.csv").squeeze()

In [7]:
xgb = XGBRegressor(
    n_estimators=100,
    max_depth=8,
    min_child_weight=5,
    n_jobs=-1,
    random_state=42
)

start = time.time()
xgb.fit(X_train, y_train)
training_time = time.time() - start

start = time.time()
xgb.fit(X_train, y_train)
training_time = time.time() - start

In [8]:
y_pred = xgb.predict(X_test)

rmse = root_mean_squared_error(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print(f"RMSE          : {rmse:.4f}")
print(f"MAE           : {mae:.4f}")
print(f"R²            : {r2:.4f}")
print(f"Training time : {training_time:.2f}s")

RMSE          : 8.7818
MAE           : 4.4337
R²            : 0.9137
Training time : 1.12s


XGBoost, although considered a far superior model compared to Linear Regression gives larger error. Lets tune the hyperparams.

In [9]:
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=3)

param_dist = {
    "max_depth": [3, 4, 6, 8, 10],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [100, 300, 500],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.3],
    "reg_alpha": [0, 0.01, 0.1],
    "reg_lambda": [1, 5, 10]
}

search = RandomizedSearchCV(
    estimator=XGBRegressor(
        objective="reg:squarederror",
        n_jobs=-1,
        random_state=42
    ),
    param_distributions=param_dist,
    n_iter=100,
    scoring="neg_root_mean_squared_error",
    cv=tscv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

start = time.time()
search.fit(X_train, y_train)
training_time = time.time() - start

print("Best params:", search.best_params_)

xgb = search.best_estimator_

y_pred = xgb.predict(X_test)

rmse = root_mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE          : {rmse:.4f}")
print(f"MAE           : {mae:.4f}")
print(f"R²            : {r2:.4f}")
print(f"Training time : {training_time:.2f}s")

Fitting 3 folds for each of 100 candidates, totalling 300 fits
Best params: {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.1, 'colsample_bytree': 0.6}
RMSE          : 9.9390
MAE           : 4.9206
R²            : 0.8895
Training time : 208.14s


Even worse result with randomized search. This is because the model is validated against data from the train set itself and it is tested against test set. The parameters best for validation set is not the best for test set.